In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -U kaggle -q

import os
os.environ["KAGGLE_API_TOKEN"] = "DIEN_TOKEN_KAGGLE_CUA_BAN"  # thay bằng token cá nhân

!kaggle datasets download -d duongtran1909/vietnamese-vehicles-dataset -p /content/data

In [ ]:
import zipfile, os, glob

zip_path = glob.glob("/content/data/*.zip")[0]
os.makedirs("/content/traffic_images", exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    all_files = [f for f in z.namelist() if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    day   = [f for f in all_files if 'day' in f.lower()][:20]
    night = [f for f in all_files if 'night' in f.lower()][:10]
    for f in day + night:
        z.extract(f, "/content/traffic_images")

imgs = glob.glob("/content/traffic_images/**/*.jp*g", recursive=True)
print(f"Đã giải nén {len(imgs)} ảnh")
print(imgs[:3])

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import glob

model = YOLO("yoloe-26l-seg.pt")

# Từ vựng mở: chỉ định đối tượng bằng ngôn ngữ tự nhiên
names = ["person", "motorcycle"]
model.set_classes(names, model.get_text_pe(names))

imgs = glob.glob("/content/traffic_images/**/*.jp*g", recursive=True)

# Chạy trên 5 ảnh đầu, lưu kết quả vào runs/segment/predict
results = model.predict(imgs[:5], save=True)

# Hiện ảnh đầu tiên ngay trong Colab
results[0].show()

# In số đối tượng phát hiện được trên từng ảnh
for i, r in enumerate(results):
    print(f"Ảnh {i+1}: {len(r.boxes)} đối tượng, thời gian suy luận {r.speed['inference']:.1f} ms")

In [ ]:
for r in results:
    r.show()

In [ ]:
from ultralytics import YOLO
import glob

imgs = glob.glob("/content/traffic_images/**/*.jp*g", recursive=True)
anh_test = imgs[0]  # đổi index nếu muốn dùng ảnh khác

# Chạy text prompt để tìm 1 xe máy, lấy box của nó làm "mẫu"
m = YOLO("yoloe-26l-seg.pt")
names = ["motorcycle"]
m.set_classes(names, m.get_text_pe(names))
r = m.predict(anh_test, conf=0.1, imgsz=1280)[0]

box_mau = r.boxes.xyxy[0].cpu().numpy()  # lấy box đầu tiên
print("Tọa độ box mẫu (x1,y1,x2,y2):", box_mau)

In [ ]:
import numpy as np
from ultralytics import YOLO
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

model = YOLO("yoloe-26l-seg.pt")
visual_prompts = dict(
    bboxes=np.array([box_mau]),   # 1 ví dụ xe máy
    cls=np.array([0])
)

results = model.predict(
    anh_test,
    visual_prompts=visual_prompts,
    predictor=YOLOEVPSegPredictor,
    save=True, conf=0.1, imgsz=1280
)
results[0].show()
print(f"Tìm được {len(results[0].boxes)} đối tượng tương tự từ 1 ví dụ mẫu")

In [ ]:
model_pf = YOLO("yoloe-26l-seg-pf.pt")  # bản prompt-free, từ vựng 4.585 lớp

results_pf = model_pf.predict(imgs[:5], save=True, imgsz=1280)

for r in results_pf:
    r.show()

# Thống kê: mô hình TỰ ĐỘNG phát hiện được những lớp gì?
from collections import Counter
dem = Counter()
for r in results_pf:
    for c in r.boxes.cls:
        dem[r.names[int(c)]] += 1
print("Các lớp phát hiện tự động:", dict(dem))